# MoneySpeaks — wav2vec2 Deepfake Detector Training

Two-stage fine-tuning on real data:
- **Real speech**: LibriSpeech dev-clean (~350MB)
- **Fake speech**: ElevenLabs-generated clips

Runtime: ~50 minutes on T4 GPU

## 1. Setup

In [ ]:
!pip install -q torch torchaudio transformers elevenlabs

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# SET YOUR ELEVENLABS API KEY HERE
import os
os.environ['ELEVENLABS_API_KEY'] = ''  # <-- PASTE YOUR KEY

## 2. Download Real Speech (LibriSpeech dev-clean)

In [ ]:
import os, subprocess, glob
from pathlib import Path

DATA_DIR = Path('/content/data')
REAL_DIR = DATA_DIR / 'real'
FAKE_DIR = DATA_DIR / 'fake'
REAL_DIR.mkdir(parents=True, exist_ok=True)
FAKE_DIR.mkdir(parents=True, exist_ok=True)

# Download LibriSpeech dev-clean (~350MB)
!wget -q --show-progress -c https://www.openslr.org/resources/12/dev-clean.tar.gz -P /content/
!tar -xzf /content/dev-clean.tar.gz -C /content/

# Collect FLAC files as real speech samples
real_files = glob.glob('/content/LibriSpeech/dev-clean/**/*.flac', recursive=True)
print(f'Found {len(real_files)} real speech files')

# Symlink into our data directory (limit to 500 for balanced training)
import random
random.shuffle(real_files)
for i, f in enumerate(real_files[:500]):
    dst = REAL_DIR / f'real_{i:04d}.flac'
    if not dst.exists():
        os.symlink(f, dst)
print(f'Linked {min(500, len(real_files))} real files')

## 3. Generate Fake Speech (ElevenLabs)

In [ ]:
import time

SENTENCES = [
    "Hello, I'm calling about your account. There's been some suspicious activity.",
    "Your social security number has been compromised. You need to act now.",
    "Hi grandma, it's me. I'm in trouble and I need your help.",
    "This is the IRS. You owe back taxes and must pay immediately.",
    "Your computer has been infected with a virus. Don't turn it off.",
    "We're calling from your bank's fraud department about unauthorized charges.",
    "Your Medicare coverage is about to expire unless you verify your information.",
    "I'm calling to confirm your appointment scheduled for next Tuesday.",
    "Hi, I wanted to check on the status of my order from last week.",
    "Good morning, this is Dr. Johnson's office calling about your test results.",
    "Your electricity will be shut off in one hour unless you make a payment.",
    "Don't tell anyone about this call. This is a confidential matter.",
    "Please hold while I transfer you to our secure verification department.",
    "We need to verify your identity. Can you confirm your date of birth?",
    "There's been a warrant issued for your arrest. This is your final notice.",
    "Your grandson was in an accident. He needs bail money right away.",
    "I'm calling about your car's extended warranty that's about to expire.",
    "Can you go to the nearest store and purchase some gift cards for me?",
    "This call is being recorded for quality assurance purposes.",
    "Hi, just calling to let you know dinner will be ready at six tonight.",
]

VOICES = {
    'voice_1': 'Ock0AL5DBkvTUDePt4Hm',  # Male
    'voice_2': 'oO7sLA3dWfQXsKeSAjpA',  # Female
    'voice_3': 'fVVjLtJgnQI61CoImgHU',  # Neutral
}

api_key = os.environ.get('ELEVENLABS_API_KEY', '')
if not api_key:
    print('WARNING: No ELEVENLABS_API_KEY set! Skipping fake audio generation.')
    print('Set it in cell 3 above and re-run.')
else:
    from elevenlabs import ElevenLabs
    client = ElevenLabs(api_key=api_key)
    
    clip_count = 0
    for voice_name, voice_id in VOICES.items():
        for i in range(50):  # 50 clips per voice = 150 total
            sentence = SENTENCES[i % len(SENTENCES)]
            filepath = FAKE_DIR / f'{voice_name}_clip_{i:03d}.mp3'
            if filepath.exists():
                clip_count += 1
                continue
            try:
                audio = client.text_to_speech.convert(
                    text=sentence, voice_id=voice_id,
                    model_id='eleven_multilingual_v2',
                )
                with open(filepath, 'wb') as f:
                    for chunk in audio:
                        f.write(chunk)
                clip_count += 1
                if clip_count % 10 == 0:
                    print(f'  Generated {clip_count} clips...')
                time.sleep(0.3)
            except Exception as e:
                print(f'  Error: {e}')
                time.sleep(2)
    
    print(f'Done! {clip_count} fake speech clips in {FAKE_DIR}')

In [ ]:
# Verify data counts
n_real = len(list(REAL_DIR.glob('*')))
n_fake = len(list(FAKE_DIR.glob('*')))
print(f'Real samples: {n_real}')
print(f'Fake samples: {n_fake}')
print(f'Total: {n_real + n_fake}')
assert n_real > 0, 'No real samples found!'
assert n_fake > 0, 'No fake samples found! Set your ELEVENLABS_API_KEY in cell 3.'

## 4. Training

In [ ]:
import random
import numpy as np
import torch
import torchaudio
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

SAMPLE_RATE = 16000
MAX_LENGTH_S = 4
MAX_SAMPLES = SAMPLE_RATE * MAX_LENGTH_S  # 64000


def apply_augmentation(waveform):
    """G.711 codec sim + noise + gain."""
    aug = waveform.clone()
    # G.711 mu-law compression (50% chance)
    if random.random() < 0.5:
        mu = 255
        sign = torch.sign(aug)
        aug = sign * torch.log1p(mu * torch.abs(aug)) / np.log(1 + mu)
        aug = sign * (torch.exp(torch.abs(aug) * np.log(1 + mu)) - 1) / mu
    # Additive noise
    if random.random() < 0.7:
        snr_db = random.uniform(-5, 20)
        signal_power = aug.pow(2).mean()
        noise_power = signal_power / (10 ** (snr_db / 10))
        aug = aug + torch.randn_like(aug) * torch.sqrt(noise_power + 1e-10)
    # Random gain
    if random.random() < 0.3:
        aug = aug * random.uniform(0.5, 1.5)
    return torch.clamp(aug, -1.0, 1.0)


class DeepfakeDataset(torch.utils.data.Dataset):
    def __init__(self, file_label_pairs, processor, augment=True):
        self.samples = file_label_pairs
        self.processor = processor
        self.augment = augment
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        # Truncate or pad to 4 seconds
        if waveform.shape[1] > MAX_SAMPLES:
            start = random.randint(0, waveform.shape[1] - MAX_SAMPLES)
            waveform = waveform[:, start:start + MAX_SAMPLES]
        elif waveform.shape[1] < MAX_SAMPLES:
            pad = torch.zeros(1, MAX_SAMPLES - waveform.shape[1])
            waveform = torch.cat([waveform, pad], dim=1)
        if self.augment:
            waveform = apply_augmentation(waveform)
        inputs = self.processor(
            waveform.squeeze().numpy(),
            sampling_rate=SAMPLE_RATE, return_tensors='pt', padding=True,
        )
        return {
            'input_values': inputs.input_values.squeeze(),
            'labels': torch.tensor(label, dtype=torch.long),
        }


# Build file lists
real_files = [(str(f), 0) for f in REAL_DIR.glob('*') if f.suffix in ('.flac', '.wav', '.mp3')]
fake_files = [(str(f), 1) for f in FAKE_DIR.glob('*') if f.suffix in ('.flac', '.wav', '.mp3')]
print(f'Real: {len(real_files)}, Fake: {len(fake_files)}')

# Balance the dataset: undersample the larger class
min_count = min(len(real_files), len(fake_files))
random.shuffle(real_files)
random.shuffle(fake_files)
all_files = real_files[:min_count] + fake_files[:min_count]
print(f'Balanced dataset: {len(all_files)} samples ({min_count} per class)')

# Train/val split (90/10)
random.shuffle(all_files)
split = int(0.9 * len(all_files))
train_files = all_files[:split]
val_files = all_files[split:]
print(f'Train: {len(train_files)}, Val: {len(val_files)}')

### Stage 1: Train classification head (transformer frozen)

In [ ]:
processor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    'facebook/wav2vec2-base', num_labels=2,
    problem_type='single_label_classification',
)

# Freeze all transformer layers
for param in model.wav2vec2.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Stage 1 — Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

train_dataset = DeepfakeDataset(train_files, processor, augment=True)
val_dataset = DeepfakeDataset(val_files, processor, augment=False)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10 * len(train_loader))

print(f'Training on {device} — {len(train_loader)} batches/epoch')
print()

for epoch in range(10):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in train_loader:
        inputs = batch['input_values'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_values=inputs, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct += (outputs.logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch['input_values'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_values=inputs)
            val_correct += (outputs.logits.argmax(-1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = correct / max(total, 1)
    val_acc = val_correct / max(val_total, 1)
    print(f'Epoch {epoch+1:2d}/10 — Loss: {total_loss/len(train_loader):.4f} | Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}')

print('\nStage 1 complete!')

### Stage 2: Fine-tune top 4 transformer layers

In [ ]:
# Unfreeze top 4 transformer layers
n_layers = len(model.wav2vec2.encoder.layers)  # 12
for param in model.wav2vec2.parameters():
    param.requires_grad = False
for i in range(n_layers - 4, n_layers):  # layers 8, 9, 10, 11
    for param in model.wav2vec2.encoder.layers[i].parameters():
        param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True
for param in model.projector.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Stage 2 — Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=0.01)

for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in train_loader:
        inputs = batch['input_values'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_values=inputs, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch['input_values'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_values=inputs)
            val_correct += (outputs.logits.argmax(-1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = correct / max(total, 1)
    val_acc = val_correct / max(val_total, 1)
    print(f'Epoch {epoch+1:2d}/5  — Loss: {total_loss/len(train_loader):.4f} | Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}')

print('\nStage 2 complete!')

## 5. Quick Test

In [ ]:
# Test on a few samples
model.eval()
print('=== Quick test ===')
test_samples = val_files[:10]
with torch.no_grad():
    for path, label in test_samples:
        waveform, sr = torchaudio.load(path)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        waveform = waveform[:, :64000]
        inputs = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors='pt')
        logits = model(inputs.input_values.to(device)).logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred = 'FAKE' if probs[1] > 0.5 else 'REAL'
        actual = 'FAKE' if label == 1 else 'REAL'
        match = '✓' if pred == actual else '✗'
        print(f'  {match} pred={pred} actual={actual} | real={probs[0]:.3f} fake={probs[1]:.3f} | {Path(path).name}')

## 6. Save & Download Checkpoint

In [ ]:
# Save checkpoint
save_path = '/content/deepfake_wav2vec2.pt'
torch.save(model.state_dict(), save_path)
size_mb = os.path.getsize(save_path) / 1024 / 1024
print(f'Checkpoint saved: {save_path} ({size_mb:.1f} MB)')

# Verify it has the right keys
ckpt = torch.load(save_path, map_location='cpu')
has_classifier = any('classifier' in k for k in ckpt.keys())
has_projector = any('projector' in k for k in ckpt.keys())
print(f'Has classifier weights: {has_classifier}')
print(f'Has projector weights: {has_projector}')
print(f'Total keys: {len(ckpt)}')

In [ ]:
# Download the checkpoint
from google.colab import files
files.download('/content/deepfake_wav2vec2.pt')